# Ingestão Automatizada - Base de Vulnerabilidade Social (VIS/SAGI)

Este notebook realiza a extração automatizada dos dados públicos de Vulnerabilidade Social do portal VIS/SAGI do Ministério da Cidadania (`aplicacoes.cidadania.gov.br`), processa a carga JSON embutida na página e carrega os registros diretamente na tabela da Camada Bronze: `vanehay.1IAST_Fase2.bronze_vulnerabilidade_social`.

In [ ]:
# Configuração e importação das bibliotecas para ingestão e carga no BigQuery
import json
import re
import urllib.request
from google.cloud import bigquery
import pandas as pd

print("Ambiente inicializado e bibliotecas importadas com sucesso para carga BigQuery.")

## 1. Extração dos Dados da URL Pública (VIS/SAGI)

Acessamos a URL de consulta do portal e extraímos a string JSON embutida na variável JavaScript `strJsondados`.

In [ ]:
# Extração resiliente com timeout de 30s e headers completos para navegadores
url = "https://aplicacoes.cidadania.gov.br/vis/data3/v.php?q[]=oNOclsLerpibuKep3bV%2Bgmxd05Kv2rmg2a19Zm91ZW2maX6JaV2Jl2Cac2CNrMmvXYt2jb5ZpJZ8e3fEosupmNDeslx8tpSg2qasw6mPuM%2BUya%2BZzd6sa2x0ZWOZZG2xpo7DxqbNolud6ayanbWUr%2BubrryYjMnHo82cao3afmhsdGVjtmlton9ypYGBv4l%2FfcCZh4FomKnapbLBmpJ%2FoaHJo5TK2sKmnqmjm9irs76ajI6RX5pmU4ibsKOdtJqt3J51jqWMvcKgybKlv9y7lZu6m6rcmIR%2Bll6LkV%2BaZlOiyZFVuYObm%2BWssomdjsPUmKV4mb7nwJl3g6iv5lzIb3puqqZTwYV4q5uwo520mq3cnnWOpYy9wqDJsqW%2F3LuVm7qbqtyYfoJnjImYZZZtXH2mbZerqaGf7JyydnebtseU15yoz92uop2np6DpnKyFZ1mHilOVXZbM3LmZr6uaYrmnrLSYmrbWpcyeob7av5qsq5RxqZh%2BgmdZh4pwml2HpcCbVIqdgYaZfpmhfE260JTWoqbA4HV0qqebm%2BaYwsCZjsXCktyjo8Dafmhsp2dxq2V9d1dYd8Siy6mY0N6yXHy2lKDapqzDqY%2B4z5TJr5nN3qxrbHRlY5lkbbGmjsPGps2iW53prJqdtZSv65uuvJiMycejzZxqjdp%2BaGx0ZWOZfpuSWKqSx5TWsJiY4a6gr61wdd%2BaucGcaJLUqNdgobzhrqGbvaec2qeuramTx8SSnHRlmOGuoK%2BtcKDapcCzcmi9wp%2Fdom6Y7sKhuJmpnt5ZsbNXkcbOnM0A4MnkvKdcraJa5aiwr6NNzNOVy6uifd68oVyulqc85rm3mKB3yqHdoKXG766nXLakWryasa%2BqocnQUy3XocbevFShtVWt4q3Cr%2FrUGgSiiqGYfeu8lq6tr5ulWcCznqLFxaKKnlPD3LasnWiZqZmJv72en7jOlIp%2FosnurlSCqaL9JqW2r2FQqNWXz12Xwpuxo6mxmP0mpba9qk28zlPWrJa%2B522prqqWqOhZsL2kTb3CoC3qn8bcwFSltqid66LBr6pNxdBTrZ6Xvu7Bpqto%2BNTnorC9V5DGzlPcoqHB3G2kobpVndqptsKYTcTGod2en33cwfflaKKf4qhtwZiZGgKl06xgyj76oqW1pFqhibywqZLRwlOVXXW%2B5MWVXLqaqN2adm6gm8rEpdOxlNCbu6Nci5aePNO7t5qcerKnzqJTweBtmKu1np085rm3pqB3xqCKqaLA3LlUsbqXm%2BeobbGmmnenlNcA4MnkrqdcaJ6o7Jy%2Ft6uOyoGh2V12vt%2Bup7C6pFo807u3mpx3xKLXXaXC6bGVXLiarJmcrr6gobiBoM%2Brpr7nbZWfsaKbmZ2ybqSSwNBT3Z6fIBy%2Fnat1ov0mp7a7pleBi1PTq6bA7baonbtVqOhZkK%2Bb8PHPnM2sr822ybC4emVsrGZ9gWRdiLVjmndjjbV9ZJaD"

print("Acessando portal VIS/SAGI...")
req = urllib.request.Request(url, headers={
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7'
})

with urllib.request.urlopen(req, timeout=30) as resp:
    html_content = resp.read().decode('latin-1', errors='ignore')

match = re.search(r"var strJsondados = \'(\[.*?\])\';", html_content, re.S)
if match:
    raw_json = match.group(1).replace('\\/', '/')
    dados = json.loads(raw_json)
    df_vulnerabilidade = pd.DataFrame(dados)
    print(f"Extração bem-sucedida! Total de registros lidos: {len(df_vulnerabilidade)}")
else:
    raise ValueError("Não foi possível localizar o payload JSON embutido na página HTML.")

## 2. Padronização do DataFrame e Validação de Colunas

Ajustamos os nomes das colunas e verificamos as primeiras linhas do dataset de Vulnerabilidade Social.

In [ ]:
# Padronização e enriquecimento de colunas (preenchendo sigla_uf com 'BR' para o agregado nacional)
registros_limpos = []
for d in dados:
    cod = str(d.get('codigo', '')).strip()
    nome = str(d.get('nome', '')).strip()
    sigla = str(d.get('sigla', '')).strip()
    
    if cod == '0' or nome.lower() == 'brasil':
        sigla = 'BR'
        
    r = {
        'codigo_ibge': cod,
        'nome_territorio': nome,
        'sigla_uf': sigla,
        'mes_ano': str(d.get('mes_ano', '')),
        'mes_ano_formatado': str(d.get('mes_ano_formatado', '')),
        'qtd_familias_extrema_pobreza': float(list(d.values())[5]) if len(d) > 5 and list(d.values())[5] is not None else 0.0,
        'qtd_familias_baixa_renda': float(list(d.values())[6]) if len(d) > 6 and list(d.values())[6] is not None else 0.0,
        'qtd_familias_acima_meio_salario': float(list(d.values())[7]) if len(d) > 7 and list(d.values())[7] is not None else 0.0
    }
    registros_limpos.append(r)

df_vulnerabilidade = pd.DataFrame(registros_limpos)
print("Colunas normalizadas e enriquecidas:", df_vulnerabilidade.columns.tolist())
display(df_vulnerabilidade.head(5))

## 3. Carga Direta na Camada Bronze do BigQuery

Enviamos os dados extraídos diretamente para a tabela da Camada Bronze no BigQuery: `vanehay.1IAST_Fase2.bronze_vulnerabilidade_social`.

In [ ]:
# Carga na tabela Bronze do BigQuery via API nativa ou DataFrame
table_id = "vanehay.1IAST_Fase2.bronze_vulnerabilidade_social"
client = bigquery.Client(project="vanehay")

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    autodetect=True
)

print(f"Carregando {len(df_vulnerabilidade)} registros em {table_id}...")
try:
    job_config.source_format = bigquery.SourceFormat.PARQUET
    job = client.load_table_from_dataframe(df_vulnerabilidade, table_id, job_config=job_config)
except Exception:
    job_config.source_format = None
    registros = df_vulnerabilidade.to_dict(orient='records')
    job = client.load_table_from_json(registros, table_id, job_config=job_config)

job.result()  # Aguarda a conclusão da carga

table = client.get_table(table_id)
print(f"SUCESSO! Tabela {table_id} materializada no BigQuery com {table.num_rows} linhas e {len(table.schema)} colunas.")

## Resumo da Execução da Ingestão

### Q&A
- **Como o dado foi extraído?** O payload JSON foi capturado diretamente da variável de memória do portal VIS/SAGI, decodificado com `latin-1` e estruturado em DataFrame.
- **Para qual tabela o dado foi enviado?** Para a tabela da Camada Bronze `vanehay.1IAST_Fase2.bronze_vulnerabilidade_social` em modo `WRITE_TRUNCATE`.

### Data Analysis Key Findings
- A extração automatizada eliminou a necessidade de arquivos CSV estáticos intermediários.
- A tipagem do BigQuery autodetectou as propriedades de vulnerabilidade social prontas para serem tratadas e normalizadas pela Camada Prata no Dataform (`silver_vulnerabilidade_social`).

### Insights or Next Steps
- Acionar em seguida o comando de compilação e execução do **Google Cloud Dataform** para processar a camada Prata e cruzar as métricas sociais com os indicadores de alfabetização do INEP.